# 5-Year Treasury Yield ETL Pipeline

## Purpose
This notebook implements an automated ETL pipeline that:
- Fetches daily 5-year U.S. Treasury yield data from Alpha Vantage API
- Performs data quality checks and standardization
- Upserts (inserts or updates) data into a Delta Lake table
- Maintains audit columns (CreatedAt, UpdatedAt) for data lineage

## Architecture
- **Source**: Alpha Vantage Financial API (CSV format)
- **Target**: Delta table `thekitchen.av.five_year_treasury`
- **Pattern**: Incremental upsert based on timestamp (SCD Type 1)
- **Compute**: Databricks Serverless (Python + PySpark)

## Workflow
1. Read current table state
2. Fetch latest data from API
3. Transform and standardize (pandas → PySpark)
4. Merge into Delta table (upsert logic)
5. Verify results

In [0]:
# ==============================================================================
# STEP 1: Read current Delta table state
# ==============================================================================
# Purpose: Verify existing data before API call to understand current coverage
# Table schema: timestamp (date), value (double), CreatedAt, UpdatedAt (timestamp)

df = spark.table("thekitchen.av.five_year_treasury")

# Display most recent records first to see latest data point
display(df.orderBy("timestamp", ascending=False))

In [0]:
# ==============================================================================
# STEP 2: Fetch data from Alpha Vantage API
# ==============================================================================
# API: Alpha Vantage TREASURY_YIELD endpoint
# Returns: CSV with columns [timestamp, value]
# Coverage: Historical + latest daily 5-year treasury yields
# Note: API key should be stored in Databricks secrets for production

import requests

# API key for Alpha Vantage (use dbutils.secrets.get() in production)
api_key = ""

# Define API parameters
function = "TREASURY_YIELD"  # Treasury yield endpoint
interval = "daily"            # Daily frequency
maturity = "5year"            # 5-year maturity
datatype = "csv"              # CSV format for easy parsing

# Construct the API URL
url = f"https://www.alphavantage.co/query?function={function}&interval={interval}&maturity={maturity}&datatype={datatype}&apikey={api_key}"

# Make the HTTP GET request
r = requests.get(url)
data = r.text

# Preview first 5 lines to verify data format
print("\n".join(data.splitlines()[:5]))

In [0]:
# ==============================================================================
# STEP 3: Transform and standardize data
# ==============================================================================
# Process: Pandas (data cleaning) → PySpark (distributed processing + audit columns)
# Data quality: Handle missing values (".") and type conversions

import pandas as pd
import io
from pyspark.sql import functions as F

# Parse CSV string into pandas DataFrame
df = pd.read_csv(io.StringIO(data))

# Data cleaning: Replace API's "." placeholder with None (missing values)
df["value"] = df["value"].replace(".", None)

# Standardize timestamp format to date type (remove time component)
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce").dt.date

# Convert yield value to numeric, coerce errors to NaN for safety
df["value"] = pd.to_numeric(df["value"], errors="coerce")

# Convert pandas → PySpark for distributed processing
spark_df = spark.createDataFrame(df)

# Add audit columns for data lineage
# CreatedAt: Set to current CET timestamp (truncated to seconds)
# UpdatedAt: Initially NULL, will be set on updates during merge
spark_df = spark_df.withColumn(
    "CreatedAt",
    F.date_trunc(
        "second", F.from_utc_timestamp(F.current_timestamp(), "Europe/Berlin")
    ),
).withColumn("UpdatedAt", F.lit(None).cast("timestamp"))

In [0]:
# ==============================================================================
# STEP 4: Upsert into Delta Lake table (SCD Type 1)
# ==============================================================================
# Pattern: MERGE operation (efficient upsert)
# Match key: timestamp (business key)
# Logic:
#   - WHEN MATCHED: Update value + UpdatedAt (revised data)
#   - WHEN NOT MATCHED: Insert new record with CreatedAt
# Benefits: ACID transactions, time travel, optimized performance

from delta.tables import DeltaTable
from pyspark.sql.functions import current_timestamp, from_utc_timestamp, date_trunc, lit

# Target Delta table in Unity Catalog
target_table = "thekitchen.av.five_year_treasury"
delta_table = DeltaTable.forName(spark, target_table)

# Prepare CET timestamp for audit columns
cet_timestamp = date_trunc(
    "second", from_utc_timestamp(current_timestamp(), "Europe/Berlin")
)

# Execute MERGE operation with error handling
try:
    delta_table.alias("target").merge(
        source=spark_df.alias("source"),
        condition="target.timestamp = source.timestamp"  # Join on business key
    ).whenMatchedUpdate(
        # Update existing records (e.g., revised treasury yields)
        set={"value": "source.value", "UpdatedAt": cet_timestamp}
    ).whenNotMatchedInsert(
        # Insert new records (new trading days)
        values={
            "timestamp": "source.timestamp",
            "value": "source.value",
            "CreatedAt": cet_timestamp,
            "UpdatedAt": lit(None).cast("timestamp"),
        }
    ).execute()
    print("Merge operation completed successfully.")
except Exception as e:
    print(f"Merge operation failed: {e}")

In [0]:
# ==============================================================================
# STEP 5: Verify merge results
# ==============================================================================
# Display updated table to confirm:
# - New records were inserted (check latest timestamps)
# - Existing records were updated (check UpdatedAt column)
# - Data quality maintained (no nulls in critical columns)

df_updated = spark.table("thekitchen.av.five_year_treasury").orderBy("timestamp", ascending=False)
display(df_updated)

---

## Technical Highlights

### Design Patterns
* **Idempotency**: Safe to re-run - merge handles duplicates gracefully
* **Incremental Load**: Only processes new/changed data via upsert logic
* **Audit Trail**: CreatedAt/UpdatedAt columns for lineage and change tracking
* **Error Handling**: Try-except blocks with descriptive error messages

### Delta Lake Benefits
* ACID transactions ensure data consistency
* Time travel capability for historical queries
* Schema evolution support for future enhancements
* Optimized merge performance for large-scale updates

### Production Readiness Checklist
- [ ] Store API key in Databricks Secrets (`dbutils.secrets.get()`)
- [ ] Add data validation (null checks, range validation for yields)
- [ ] Implement logging (e.g., MLflow or custom logs)
- [ ] Schedule via Databricks Jobs (e.g., daily at market close)
- [ ] Add alerting for API failures or data quality issues
- [ ] Consider OPTIMIZE and VACUUM for table maintenance

### Key Metrics
* **Data Freshness**: Daily updates aligned with market data
* **Historical Coverage**: Complete 5-year treasury yield history
* **Data Quality**: Null handling + type validation ensures clean data